# 📓 Update dataset with last boxscores and matches

# Please first run full pipeline until merge_clean_seasons_boxscores to have base data to work with and merge

In [1]:

import os
import pandas as pd
from datetime import datetime
from src.config import *
from src.utils import *
from src.nba_scrapping import *



In [2]:
# ⚙️ Initialisation du run
run_timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
current_season = '2025-26'  # À rendre dynamique si besoin plus tard


In [3]:

# 📁 Préparation des dossiers de sortie
os.makedirs(DATA_LAST_GAMES_DIR, exist_ok=True)
os.makedirs(DATA_LAST_BOXSCORES_BATCHES_DIR, exist_ok=True)

# 🔄 Chargement des historiques si existants
hist_games_path = get_latest_file(DATA_LAST_GAMES_MERGED_DIR)


In [4]:

# 📥 1. Téléchargement des matchs de la saison actuelle
print("\n📥 Téléchargement des nouveaux matchs pour la saison:", current_season)
matchs_output_dir = os.path.join(DATA_LAST_GAMES_DIR, current_season)
last_games_path = download_games_for_seasons([current_season], matchs_output_dir, run_timestamp, max_retries=10)


📥 Téléchargement des nouveaux matchs pour la saison: 2025-26
Extraction saison 2025-26


In [5]:

# 📊 2. Comparaison avec les données historiques pour trouver les nouveaux matchs
games_to_scrape = get_new_games(hist_games_path, last_games_path)
print(f"✅ {len(games_to_scrape)} nouveaux matchs trouvés à scraper.")

if games_to_scrape.empty:
    print("✅ Aucun nouveau match à scraper. Fin du script.")
    exit(0)

0 nouveaux matchs à traiter
✅ 0 nouveaux matchs trouvés à scraper.
✅ Aucun nouveau match à scraper. Fin du script.


In [6]:
games_to_scrape

,SEASON_ID,TEAM_ID,TEAM_ABBREVIATION,TEAM_NAME,GAME_ID,GAME_DATE,MATCHUP,WL,MIN,PTS,...,OREB,DREB,REB,AST,STL,BLK,TOV,PF,PLUS_MINUS,SEASON


In [ ]:

# 💾 3. Merge historique + nouveaux matchs
historical_games = pd.read_csv(hist_games_path, low_memory=False, dtype={'GAME_ID': str})
all_games = pd.concat([historical_games, games_to_scrape], ignore_index=True)

games_output_path = save_dataframe_to_csv(all_games, DATA_LAST_GAMES_MERGED_DIR, 'games_merged_all_seasons', run_timestamp)
print(f"✅ Jeux de matchs fusionnés sauvegardés dans {games_output_path}.")


✅ Jeux de matchs fusionnés sauvegardés dans data/01_bronze/games/games_merged_all_seasons_2025-11-12_15-40-19.csv.


: 

In [ ]:

# 🏀 4. Scraping des nouveaux boxscores
print(f"--- Traitement de la saison {current_season} ---")
season_df = games_to_scrape[games_to_scrape['SEASON'] == current_season]
season_output_dir = os.path.join(DATA_LAST_BOXSCORES_BATCHES_DIR, run_timestamp, current_season)
scrape_boxscores_v3_for_games(season_df, season_output_dir,max_retries=10)

--- Traitement de la saison 2025-26 ---
[DEBUG] Found 0 batch files for endpoint 'traditional' in season 2025-26
[DEBUG] Found 0 batch files for endpoint 'advanced' in season 2025-26
[DEBUG] Found 0 batch files for endpoint 'fourfactors' in season 2025-26
[DEBUG] Found 0 batch files for endpoint 'misc' in season 2025-26
[DEBUG] Found 0 batch files for endpoint 'scoring' in season 2025-26
[DEBUG] Found 0 batch files for endpoint 'usage' in season 2025-26
--------- 12 GAME_ID to scrap for season 2025-26 ---------
[1/12] GAME_ID: 0022500178 - 2025-11-05
[2/12] GAME_ID: 0022500174 - 2025-11-05
[3/12] GAME_ID: 0022500179 - 2025-11-05
[4/12] GAME_ID: 0022500177 - 2025-11-05
[5/12] GAME_ID: 0022500176 - 2025-11-05
[6/12] GAME_ID: 0022500171 - 2025-11-05
[7/12] GAME_ID: 0022500173 - 2025-11-05
[8/12] GAME_ID: 0022500172 - 2025-11-05
[9/12] GAME_ID: 0022500181 - 2025-11-05
[10/12] GAME_ID: 0022500180 - 2025-11-05
[11/12] GAME_ID: 0022500175 - 2025-11-05
[12/12] GAME_ID: 0022500182 - 2025-11-06


True

In [ ]:
print("\n✅ Mise à jour complète terminée. Données prêtes pour le feature engineering.")



✅ Mise à jour complète terminée. Données prêtes pour le feature engineering.
